# 🔧 How LM Studio *Selects*, *References*, and *Calls* Your Python Tools

### Dinesh AI Academy | Day 4 — Agents & MCP (Deep Dive · LM Studio Edition)

**Learning objective:**
By the end of this notebook you will be able to draw, from memory, the exact
sequence of events that happens between "the model decides to use a tool" and
"a line of your Python code actually runs" — and say precisely **who** does
**what**, and **on which machine**, at every single step.

**Why this notebook exists:** the [Building AI Agents](./1b-Building_AI_Agents_LM_Studio.ipynb)
notebook *used* function calling to build an agent loop, against a local LM
Studio model. This notebook opens the hood and looks at the machinery itself
— how a plain Python function becomes something the model can "see", how the
model decides to use it, and where the actual function execution happens.
Once this is crystal clear, agents, MCP, and every framework you'll ever use
(LangChain, ADK, OpenAI Agents SDK) stop looking like magic.

This is a twin of [2-How_Gemini_Selects_And_Calls_Tools.ipynb](./2-How_Gemini_Selects_And_Calls_Tools.ipynb),
walking through the exact same ten sections — only the model changed.

> ⚠️ **This notebook runs entirely against your local LM Studio server — no API key,
> no cloud, no cost.** Because it talks to `localhost`, it must be run **locally**
> (e.g. in VS Code), not in Google Colab. It reuses the same connection pattern as
> the Day 3 `4- Tools_LM_Studio.ipynb` notebook and the `1b` agent-loop notebook —
> if anything below is unfamiliar, those notebooks cover the basics in more depth.

## 1. The One Question That Fixes 90% of the Confusion

> ### ❓ "Does the model run my Python code?"
> ### 🚫 **No. Never. Not once. Not even a little bit.**

The model running inside LM Studio is a language model, full stop — it has
never seen your Python interpreter or your function's source code, whether
it's running on Google's servers (Gemini) or on `localhost` (LM Studio).
**All it can ever do is generate text** — and "calling a tool" is really the
model generating a very specific, structured *piece of text* that says, in
effect:

> *"If I were you, I'd now call the function named `get_weather` with the
> argument `city='Tokyo'`."*

That's it. That's the entire "call". The model's job ends the moment it
produces that structured text. **Your own code** is the thing that reads
that text, finds the real Python function, and runs it.

```text
   YOUR PYTHON PROCESS                              LM STUDIO SERVER PROCESS
   ────────────────────                              ─────────────────────────
   def get_weather(city):                              the loaded model
       ...real code...                                 (has NEVER seen the
        │                                                code above — only
        │  1. send: function NAME +                     its name + schema)
        │     SCHEMA (no source code)                   Still a SEPARATE
        ├──────────────────────────────────────────────▶ PROCESS from yours —
        │                                               just reached over
        │                                               localhost:1234 instead
        │                                               of the open internet.
        │                                               2. reads prompt + schema
        │                                               3. decides: "I should
        │                                                   request get_weather"
        │  4. returns: plain TEXT that says
        │     {"name": "get_weather",
        │      "arguments": "{\"city\": \"Tokyo\"}"}
        ◀──────────────────────────────────────────────┤
        │
        │  5. YOUR code reads that text,
        │     finds the REAL function, and
        │     calls it:  get_weather("Tokyo")
        │     ← this is the ONLY line in this
        │       entire diagram that executes code
        │
        │  6. send the RESULT back
        ├──────────────────────────────────────────────▶
        │                                               7. reads the result,
        │  8. returns: final answer text                   writes an answer
        ◀──────────────────────────────────────────────┤
```

Keep this diagram in your head for the rest of the notebook — every section
below is just zooming into one arrow of it. **The only thing that changed
from the Gemini version of this diagram is *where* the right-hand box lives**
— the boundary itself (only name + schema crosses over, never source code)
is identical in kind, whether that box is on Google's servers or your own
laptop.

## 2. Setup — Connect to LM Studio

Unlike Gemini, there's no API key here — LM Studio runs an **OpenAI-compatible**
server on your own machine, and the official `openai` Python SDK can talk to it
directly by pointing `base_url` at `localhost` instead of OpenAI's servers.

1. Open LM Studio and make sure a **tool-calling capable** chat model is downloaded
   (Llama 3.1+, Qwen, and Gemma 3/4 instruct families generally support it).
2. Go to the **Developer** tab and click **Start Server** (default `http://localhost:1234`).
3. Run the cells below to list what's loaded, then set `MODEL` to match exactly.

> **No rate limits, no retries needed:** since everything runs locally, there's no
> quota to worry about like on Gemini's free tier — the only failure mode is the
> server not being started yet, or the model not being loaded.

In [ ]:
# Install the OpenAI Python SDK -- LM Studio speaks the same API, so this is
# the only client library we need (no google-genai here).
%pip install -q openai

In [ ]:
import inspect
import json
from openai import OpenAI

LM_STUDIO_BASE_URL = "http://localhost:1234/v1"

# LM Studio doesn't check the API key, but the SDK requires some string.
client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key="lm-studio")

print("Models available in LM Studio:")
for m in client.models.list().data:
    print(" -", m.id)

In [ ]:
# Set this to exactly one of the model ids printed above --
# it must be a tool-calling capable model (see Section 2).
MODEL = "gemma-4-e2b-it-qat"

print("LM Studio client is ready.")
print("Model:", MODEL)

## 3. What the Model Actually Receives About a Function (Hint: Not the Function)

Here is one real Python function, exactly as you'd write it for any normal
program — nothing "AI" about it yet.

In [ ]:
def get_weather(city: str) -> dict:
    '''Get the current simulated weather for a city.

    Args:
        city: Name of the city, e.g. "Tokyo".
    '''
    WEATHER_DB = {
        "tokyo": {"temp_c": 26, "condition": "sunny"},
        "paris": {"temp_c": 18, "condition": "cloudy"},
        "mumbai": {"temp_c": 31, "condition": "humid, partly cloudy"},
    }
    data = WEATHER_DB.get(city.strip().lower(), {"temp_c": 20, "condition": "unknown"})
    return {"city": city, **data}

# Sanity check -- plain Python, no model involved.
print(get_weather("Tokyo"))

To let the model *reference* this function as a tool, we don't send it the
function. We hand-write a small JSON object — a **tool schema** — that
describes its *interface*: a name, a plain-English description, and the
shape of its arguments. LM Studio (like OpenAI, and unlike Gemini's own SDK)
expects this wrapped one level deeper, as
`{"type": "function", "function": {name, description, parameters}}`:

In [ ]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Gets the current simulated weather (temperature in Celsius, condition) for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Tokyo'."}
            },
            "required": ["city"]
        }
    }
}

# THIS -- and only this -- is what crosses the wire to the LM Studio server.
print(json.dumps(weather_tool, indent=2))

> **Read that JSON out loud in class.** There is no `if/elif`, no
> `WEATHER_DB`, no return statement anywhere in it. The model will never know
> *how* `get_weather` computes its answer, that it's fake/simulated data, or
> that it even exists as Python — only that a capability with this name,
> this description, and this argument shape is *available* to be requested.
>
> **This is also a security boundary worth saying explicitly:** your source
> code, business logic, database credentials embedded in a tool's
> implementation, etc. never leave your machine — not even to a *local*
> server process. Only the interface does.

## 4. Two Ways a Python Function Becomes a "Tool" Schema

### 4a. Manual — you write the JSON schema by hand
That's what we just did above. You are responsible for keeping the schema's
`parameters` in sync with the function's real signature — if you add a
parameter to `get_weather` and forget to update the JSON, the model will
never know that parameter exists.

### 4b. Semi-automatic — build the schema yourself, by introspection
Gemini's `google-genai` SDK has a built-in trick: pass it a **raw Python
function** and it inspects the function's signature + docstring to build the
schema *for you*, automatically. **The `openai` SDK (and therefore LM
Studio) has no equivalent feature** — there's no way to pass a raw callable
into `tools=[...]` and have it dispatch itself. You always build the JSON
schema yourself, and you always write the dispatch loop yourself (Section 9
comes back to why that's actually a good thing).

That said, the *information* the schema needs — name, parameter names/types,
a description — still lives entirely inside the function's signature and
docstring. Nothing stops you from writing your **own** small introspection
helper to generate it, the same way Gemini's SDK does internally:

In [ ]:
def function_to_tool_schema(func) -> dict:
    '''A tiny DIY version of what google-genai's SDK does automatically:
    build an OpenAI-style tool schema from a function's signature + docstring.
    Deliberately simple -- assumes every parameter is a plain str/int/float and
    is required. A real implementation would map full type hints and defaults.
    '''
    TYPE_MAP = {str: "string", int: "integer", float: "number", bool: "boolean"}
    sig = inspect.signature(func)
    properties = {
        name: {"type": TYPE_MAP.get(p.annotation, "string")}
        for name, p in sig.parameters.items()
    }
    description = (func.__doc__ or "").strip().splitlines()[0] if func.__doc__ else ""
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": list(sig.parameters),
            },
        },
    }

generated = function_to_tool_schema(get_weather)
print(json.dumps(generated, indent=2))
print("\nMatches our hand-written schema exactly?", generated == weather_tool)

Probably `False` — and that's the point, not a bug. Our hand-written version
in Section 3 added a per-*property* description (`"City name, e.g. 'Tokyo'."`)
and used a differently-worded top-level description; the DIY introspector
only pulled the docstring's first line and skipped per-argument descriptions
entirely (a real limitation of this simple version — a fuller one could parse
`Args:` lines too). **Same underlying idea, two different amounts of
polish** — exactly the tradeoff you're making any time you choose "hand-write
it carefully" vs. "generate it automatically and move on."

| | Manual (4a) | Introspected, DIY (4b) |
|---|---|---|
| Who writes the schema? | You, by hand | A helper function you write once, reused for every tool |
| Source of the description | Whatever string you type | Your docstring's first line (so **write good docstrings**) |
| Built into the SDK? | N/A | **No** — unlike `google-genai`, `openai`/LM Studio has no automatic version |
| Risk of drift | High — easy to forget to update | Low — schema always matches the real signature |

**Either way, the destination is identical**: a small JSON object containing
a name, a description, and a parameter schema — nothing else ever leaves
your machine, whether "leaving" means crossing the internet to Google or
crossing a localhost socket to LM Studio.

## 5. How the Model "Selects" a Tool — There Is No `if` Statement

It's tempting to imagine the model has some internal `switch (userIntent)`
statement that routes to a tool. **It doesn't.** The model is doing the exact
same thing it always does — predicting the next tokens — except it has been
trained so that when the *conversation + the list of available tool schemas*
make a function call the most likely continuation, it emits a structured
tool call (name + JSON arguments) instead of prose.

That means **tool selection is a semantic-matching problem, not a logic
problem** — and the single biggest lever you have over it is the
**`description` field**. Watch what happens with the same prompt and two
different tool descriptions.

In [ ]:
def ask(prompt: str, description: str):
    '''Send one prompt with ONE tool available, whose description we control.'''
    tool = {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        tools=[tool],
        tool_choice="auto",
    )
    message = response.choices[0].message
    if message.tool_calls:
        for tc in message.tool_calls:
            print(f"  -> requested tool: {tc.function.name}({tc.function.arguments})")
    if message.content:
        print(f"  -> answered with plain text: {message.content.strip()[:160]}")

prompt = "Is it a good day to hang laundry outside in Tokyo?"

print("Vague description:")
ask(prompt, "Gets information about a city.")

print("\nPrecise description:")
ask(prompt, "Gets the current temperature and weather condition (e.g. rainy, sunny) for a city.")

Same model, same prompt, same underlying Python function — the only thing
that changed is a sentence of English in the schema. **The description is
the entire "API documentation" the model gets to decide *whether* and *when*
your tool is relevant.** A vague description is the #1 real-world cause of
"why didn't it call my tool?" bugs — on a local model even more than on
Gemini, since smaller local models lean more heavily on that one sentence to
compensate for weaker general reasoning.

## 6. You Can Also Make the "When" Explicit Yourself — `tool_choice`

Selection isn't *purely* left to the model's judgement. Gemini calls this
`tool_config` with modes `AUTO`/`ANY`/`NONE`; the OpenAI-compatible API (and
therefore LM Studio) calls the same idea `tool_choice`:

| OpenAI/LM Studio `tool_choice` | Gemini `tool_config` equivalent | Behaviour |
|---|---|---|
| `"auto"` (default) | `AUTO` | The model decides — tool call or plain text, its choice |
| `"required"` | `ANY` | The model **must** call a tool this turn |
| `"none"` | `NONE` | The model is **not allowed** to call any tool this turn, even if one would help |
| `{"type": "function", "function": {"name": ...}}` | `ANY` + `allowed_function_names` | Force one *specific* tool (full OpenAI spec — support varies by server, see below) |

Try `"required"` on a prompt that doesn't obviously need a tool at all —
forcing a call even though nothing really fits:

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Tell me something interesting."}],
    tools=[weather_tool],
    tool_choice="required",  # force a tool call even though this prompt doesn't need one
    max_tokens=150,          # cap generation -- a forced, ill-fitting call can otherwise ramble
)
message = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("tool_calls:   ", [(tc.function.name, tc.function.arguments) for tc in (message.tool_calls or [])])
print("content:      ", repr(message.content))

Whatever you just saw is a real, honest result, and there are two equally
instructive outcomes you might get:

- **A forced, nonsensical call** — e.g. treating a random word as a city
  name — because `"required"` removes the model's ability to say "none of
  these apply."
- **No tool call at all**, despite `"required"` supposedly guaranteeing one —
  a sign this particular server/model combination doesn't honor `"required"`
  perfectly. That's not a bug in your code; it's a real limit of small local
  models and their OpenAI-compatibility layers.

Either result teaches the same lesson: `"required"` is the least reliable of
the three `tool_choice` modes, especially on local models.

Now try forcing one *specific* function by name — some OpenAI-compatible
local servers support this object form, some don't yet:

In [ ]:
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
        tools=[weather_tool, {"type": "function", "function": {
            "name": "get_current_time", "description": "Get the current time for a timezone.",
            "parameters": {"type": "object", "properties": {"timezone": {"type": "string"}}, "required": ["timezone"]},
        }}],
        tool_choice={"type": "function", "function": {"name": "get_current_time"}},  # force the WRONG tool, on purpose
    )
    message = response.choices[0].message
    print("Forced call:", [(tc.function.name, tc.function.arguments) for tc in (message.tool_calls or [])])
except Exception as e:
    print("This server rejected per-function forcing:", e)
    print("-> a real gap between \"OpenAI-compatible\" and \"100% of the OpenAI spec.\"")
    print("   Always check what your specific server actually supports -- don't assume")
    print("   compatibility from the name alone.")

## 7. The Complete Call Chain — Who Calls Whom, Where

This is the table to put on a slide. Every row is one arrow from the Section
1 diagram, made explicit — with the OpenAI/LM Studio names for each piece.

| Step | Who / what does this | Runs on | What happens |
|---|---|---|---|
| 1 | **You** | Your process | Write `tools=[...]` (schema) and call `client.chat.completions.create(...)` |
| 2 | `openai` SDK | Your process | Serializes your messages + schema into an HTTP JSON request |
| 3 | LM Studio server | The LM Studio process (localhost) | Receives the request; the loaded model reads messages + schema as context |
| 4 | **The model** | The LM Studio process | Predicts a structured tool call (name + JSON `arguments` string) instead of prose — **no code executes here** |
| 5 | `openai` SDK | Your process | Deserializes the HTTP response into a Python `ChatCompletion` object |
| 6 | **Your code** | Your process | Reads `message.tool_calls[i].function.name` and `.arguments` |
| 7 | **Your code** | Your process | Looks up that name string in *your own* dispatch table (e.g. `TOOLBOX[name]`) |
| 8 | **Your code** | Your process | Calls the *real* function: `TOOLBOX[name](**args)` — 🔴 **the only step where actual code executes anywhere in this chain** |
| 9 | **Your code** | Your process | Wraps the return value in a `{"role": "tool", "tool_call_id": ..., "content": ...}` message and sends a *new* request |
| 10 | The model | The LM Studio process | Reads the result as context, predicts the final answer text |

Notice rows 6–9: **all four of those steps are your own Python code.**
Nothing about them is "AI" — they're a dictionary lookup and a function
call, the same as any plugin system you'd have written in 2015. The only
genuinely new ingredient is row 4: a model that can decide, from plain
English, *when* to ask for one. **Even though row 3 happens on the same
physical laptop as row 1 here, it's still a separate process talking over
HTTP** — the process boundary (and the fact that only the schema crosses it,
never your source) is what matters, not the physical distance.

## 8. Proving It: Two Experiments

### Experiment A — the model "asks" for a tool that doesn't exist on our side
Because the model only ever emits *text describing a request*, nothing stops
it from naming a tool your dispatch table doesn't actually have (a typo'd
schema, a hallucinated name, a tool you removed). If that happens, the
failure is entirely **on your side**, at step 7 above — the LM Studio
process is long finished with that request and doesn't know or care what
you do with it.

In [ ]:
TOOLBOX = {"get_weather": get_weather}  # deliberately does NOT contain "get_forecast"

def execute_tool(name: str, args: dict):
    '''Step 7 + 8 from the table above, isolated as one function.'''
    func = TOOLBOX.get(name)
    if func is None:
        raise ValueError(
            f"The model requested '{name}', but our TOOLBOX doesn't have it. "
            f"This is OUR error, raised on OUR machine -- the LM Studio "
            f"server's job ended the moment it sent the request."
        )
    return func(**args)

# Simulate the model having requested a tool we never registered.
try:
    execute_tool("get_forecast", {"city": "Tokyo"})
except ValueError as e:
    print("Caught locally:", e)

### Experiment B — watch steps 4 and 8 happen for real, one at a time
No hidden loop this time — we'll print a message *between* "the model asked
for the tool" and "we actually ran it", so the two moments are visibly
separate in the output.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What's the weather in Mumbai right now?"}],
    tools=[weather_tool],
    tool_choice="auto",
)

tool_call = response.choices[0].message.tool_calls[0]
print("STEP 4 (LM Studio process, just finished): model requested ->",
      tool_call.function.name, tool_call.function.arguments)

print("... nothing has executed yet. The model is done. We are now in OUR code ...")

args = json.loads(tool_call.function.arguments)
result = execute_tool(tool_call.function.name, args)
print("STEP 8 (your process, happening now):      ran the real function ->", result)

## 9. Why There's No "Automatic Function Calling" Section Here

The Gemini version of this notebook has a Section 8 showing
`google-genai`'s **automatic function calling**: pass raw Python functions
into `tools=[...]` and the SDK builds the schema *and* runs the dispatch
loop (steps 6–9) internally, handing you only the final text.

**The `openai` SDK has no equivalent.** Every LM Studio / OpenAI-compatible
integration is, by necessity, "manual mode": you always build the schema
(Section 4), and you always write the loop that reads `tool_calls`, executes
them, and sends the result back as a `role: "tool"` message — there is no
hidden version doing that for you.

This isn't a missing feature so much as a design difference worth noticing:
**it's exactly why the `1b` agent-loop notebook's `run_agent()` function has
to exist as real code you write** — with LM Studio, there was never an
"automatic" shortcut to opt out of in the first place. If you ever *do* want
a convenience wrapper (e.g. `function_to_tool_schema()` from Section 4b, or
a small library), you're building it yourself, on top of the same manual
call chain from Section 7 — never underneath it.

## 10. 🎤 Crystal-Clear Cheat Sheet — Read This Straight to the Class

| Question | Answer |
|---|---|
| Does the model run my Python code? | **No.** It only ever generates text. |
| Then who runs it? | **Your own process**, at step 8 of the Section 7 table. |
| What does the model receive about my function? | Only its **name, description, and parameter schema** — never the source code. |
| How does the model "choose" a tool? | The same way it predicts any token: by conditioning on the conversation **and the schema text** — the `description` field matters most. |
| Can I control *when* it's allowed to call a tool? | Yes — `tool_choice` with `"auto"` / `"required"` / `"none"` (and per-function forcing, on servers that support it). |
| What if the model requests a tool I never defined? | Your dispatch code raises the error, locally. The LM Studio server never finds out. |
| Do I have to hand-write the JSON schema every time? | You do, or you write your own introspection helper (Section 4b) — unlike Gemini's SDK, there's no built-in automatic version. |
| Who decides how many tool calls happen in a row? | **Always your own loop** here — see `1b-Building_AI_Agents_LM_Studio.ipynb`. |

## 11. 🧪 Classroom Challenge

For the prompt **"Convert today's temperature in Tokyo to Fahrenheit."**,
fill in this table *before* running anything — then verify with code:

| Step # (from Section 7's table) | Who does it | What exactly happens for this prompt? |
|---|---|---|
| 1 | ? | ? |
| 4 | ? | ? |
| 8 | ? | ? |
| 9 | ? | ? |
| 10 | ? | ? |

**Bonus question for discussion:** this prompt needs *two* tool calls
(`get_weather`, then a conversion). At which exact row of the Section 7
table does the *second* request for a tool get generated — and why can't
the model ask for both tools in a single row-4 moment?

*(Answer: it can request both in the same turn if it doesn't need the first
result to compute the second's arguments — Section 5 of the `1b` notebook
shows an example of exactly that. But here it does need the first result, so
row 4 for the conversion can only happen after a fresh row 3, once the
weather result has gone back in as a `role: "tool"` message. That's exactly
why the `1b` agent-loop notebook needed a loop instead of one round trip.)*

## 🎓 Day 4 Takeaway

By the end of this notebook, you should be able to explain, without
hesitating:

1. **The model never executes your code** — it only ever generates text that
   describes a request (a function name + JSON arguments) — true whether
   that model runs on Google's servers or in LM Studio on your own laptop.
2. **Only the interface crosses the process boundary** — name, description,
   parameter schema. Never the function body. That boundary matters even
   when both sides are on the same machine.
3. A tool schema can be **hand-written** (full control) or built by your
   **own introspection helper** from a raw Python function — but unlike
   Gemini's SDK, LM Studio/OpenAI has no built-in automatic version.
4. "Tool selection" is **semantic matching against schema text**, driven
   overwhelmingly by the `description` field — not a hardcoded router.
   `tool_choice` (`"auto"`/`"required"`/`"none"`) lets you steer that decision.
5. The actual function call happens in **your own process**, at one
   specific, inspectable line of your own code — and with LM Studio, that
   line is *always* something you wrote; there's no automatic mode to fall
   back on.
6. That "always manual" reality is precisely why the agent-loop notebook's
   `run_agent()` is necessary infrastructure, not an optional convenience.
7. **Model capability is part of the equation, not just the schema** — the
   same well-described tool can be selected reliably by one local model and
   missed or misused by a smaller one; always test against the specific
   model you intend to ship with.

### The one diagram to remember

```text
 YOUR PROCESS                                    LM STUDIO SERVER PROCESS
 ─────────────                                    ─────────────────────────
 real Python function  ──(name + schema only)──▶  model reads + decides
                                                          │
 YOUR code runs the      ◀──(text: name + args)──────────┘
 real function here
      │
      └──(result)──────────────────────────────▶  model reads + answers
```

## Official references

- LM Studio — Local server & OpenAI compatibility: https://lmstudio.ai/docs/app/api
- LM Studio — Tool use / function calling: https://lmstudio.ai/docs/app/api/tools
- OpenAI API — Function calling guide: https://platform.openai.com/docs/guides/function-calling
- OpenAI API reference — `tool_choice`: https://platform.openai.com/docs/api-reference/chat/create#chat-create-tool_choice
- OpenAI Python SDK: https://github.com/openai/openai-python
- Python `inspect` module (what our DIY schema introspection is built on): https://docs.python.org/3/library/inspect.html

See also: [1b-Building_AI_Agents_LM_Studio.ipynb](./1b-Building_AI_Agents_LM_Studio.ipynb)
for the full agent loop built on top of everything in this notebook,
[2-How_Gemini_Selects_And_Calls_Tools.ipynb](./2-How_Gemini_Selects_And_Calls_Tools.ipynb)
for the same ten sections against Gemini, and
[1-MCP_Server_Basics.ipynb](./1-MCP_Server_Basics.ipynb) for tools you don't
have to write the schema for at all.